In [2]:
import pandas as pd

In [3]:
#load the data_ingestor via the DataIngestorFactory
import os
import sys
from ydata_profiling import ProfileReport
sys.path.append(os.path.join(os.getcwd(), "..","src"))

from utils.data_ingestor import DataIngestorFactory

file_path = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "Audible data.zip"))
file_extension = os.path.splitext(file_path)[1]

ingestor = DataIngestorFactory.get_data_ingestor(file_extension)
df = ingestor.ingest(file_path)




c:\Users\Owner\anaconda3\envs\vsproj\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Check the shape and first few entries of the data to see if the data was loaded correctly

In [4]:
df.shape

(87489, 8)

In [5]:
df.head(20)

,name,author,narrator,time,releasedate,language,stars,price
0,Geronimo Stilton #11 & #12,Writtenby:GeronimoStilton,Narratedby:BillLobely,2 hrs and 20 mins,04-08-08,English,5 out of 5 stars34 ratings,468.00
1,The Burning Maze,Writtenby:RickRiordan,Narratedby:RobbieDaymond,13 hrs and 8 mins,01-05-18,English,4.5 out of 5 stars41 ratings,820.00
2,The Deep End,Writtenby:JeffKinney,Narratedby:DanRussell,2 hrs and 3 mins,06-11-20,English,4.5 out of 5 stars38 ratings,410.00
3,Daughter of the Deep,Writtenby:RickRiordan,Narratedby:SoneelaNankani,11 hrs and 16 mins,05-10-21,English,4.5 out of 5 stars12 ratings,615.00
4,"The Lightning Thief: Percy Jackson, Book 1",Writtenby:RickRiordan,Narratedby:JesseBernstein,10 hrs,13-01-10,English,4.5 out of 5 stars181 ratings,820.00
5,The Hunger Games: Special Edition,Writtenby:SuzanneCollins,Narratedby:TatianaMaslany,10 hrs and 35 mins,30-10-18,English,5 out of 5 stars72 ratings,656.00
6,Quest for the Diamond Sword,Writtenby:WinterMorgan,Narratedby:LukeDaniels,2 hrs and 23 mins,25-11-14,English,5 out of 5 stars11 ratings,233.00
7,The Dark Prophecy,Writtenby:RickRiordan,Narratedby:RobbieDaymond,12 hrs and 32 mins,02-05-17,English,5 out of 5 stars50 ratings,820.00
8,Merlin Mission Collection,Writtenby:MaryPopeOsborne,Narratedby:MaryPopeOsborne,10 hrs and 56 mins,02-05-17,English,5 out of 5 stars5 ratings,"1,256.00"
9,The Tyrant’s Tomb,Writtenby:RickRiordan,Narratedby:RobbieDaymond,13 hrs and 22 mins,24-09-19,English,5 out of 5 stars58 ratings,820.00


In [6]:
df.isnull().sum()

name           0
author         0
narrator       0
time           0
releasedate    0
language       0
stars          0
price          0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87489 entries, 0 to 87488
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   name         87489 non-null  object
 1   author       87489 non-null  object
 2   narrator     87489 non-null  object
 3   time         87489 non-null  object
 4   releasedate  87489 non-null  object
 5   language     87489 non-null  object
 6   stars        87489 non-null  object
 7   price        87489 non-null  object
dtypes: object(8)
memory usage: 5.3+ MB


Right away I see that release date needs to be changed to date time format, time would benefit in being in date time as well. Time is looking kind of funky as it has words and numbers and it isnt in duration format. Stars can be numeric or categorical. Price can be numeric

In [9]:
#convert releasedate to datetime
df['releasedate'] = pd.to_datetime(df['releasedate'], errors='coerce')
#covert time to time format
#df['time'] = pd.to_datetime(df['time'], errors='coerce').dt.time



C:\Users\Owner\AppData\Local\Temp\ipykernel_6356\931483824.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['releasedate'] = pd.to_datetime(df['releasedate'], errors='coerce')


In [10]:
#convert time to minutes


df["minutes"] = (
    df["time"]
    .str.extract(r"(?:(\d+)\s*hr[s]?)?\s*(?:and)?\s*(\d+)\s*min")
    .astype(float)
    .fillna(0)
    .pipe(lambda x: x[0]*60 + x[1])
)
#drop time column
df = df.drop(columns=["time"])


Language could be more consistent. We will change that to lower

In [11]:
df["language"] = df["language"].str.lower()

In [12]:
df["language"].unique()

array(['english', 'hindi', 'spanish', 'german', 'french', 'catalan',
       'swedish', 'italian', 'danish', 'finnish', 'dutch', 'hebrew',
       'russian', 'polish', 'galician', 'afrikaans', 'icelandic',
       'romanian', 'japanese', 'tamil', 'portuguese', 'urdu', 'hungarian',
       'czech', 'bulgarian', 'mandarin_chinese', 'basque', 'korean',
       'arabic', 'greek', 'turkish', 'ukrainian', 'slovene', 'norwegian',
       'telugu', 'lithuanian'], dtype=object)

Narrator looks kind of weird, it starts with "Narratedby:...". Narrated by is irrelevant, we wont need the white space either. Author is just as weird, it starts with "Writtenby:...". Written by is going to irrelevant as well.

In [13]:
#r means literal string-dont treat the backsplashes as an escape, treat literally
#^            start of string
#Narrated by  literal text
#\s*          any spaces(space optional)
#regex=True for patterns that are similar not exact

df["narrator"] = df["narrator"].str.replace(r"^Narrated\s*by:\s*", "", regex=True)
#adding space before the capital letters
#(?<=[a-z])     lookbehind for lowercase letter
#(?=[A-Z])      lookahead for uppercase letter
#in other words: between a lowercase and uppercase letter, add a space
df["narrator"] = df["narrator"].str.replace(r"(?<=[a-z])(?=[A-Z])", " ", regex=True)
df["author"] = df["author"].str.replace(r"^Written\s*by:\s*", "", regex=True)
#adding space before the capital letters
df["author"] = df["author"].str.replace(r"(?<=[a-z])(?=[A-Z])", " ", regex=True)

Price and Stars need to change type. They are currently object type and they would benefit from being float for price and stars.

In [14]:
df[["price","stars"]].head(10)

,price,stars
0,468.00,5 out of 5 stars34 ratings
1,820.00,4.5 out of 5 stars41 ratings
2,410.00,4.5 out of 5 stars38 ratings
3,615.00,4.5 out of 5 stars12 ratings
4,820.00,4.5 out of 5 stars181 ratings
5,656.00,5 out of 5 stars72 ratings
6,233.00,5 out of 5 stars11 ratings
7,820.00,5 out of 5 stars50 ratings
8,"1,256.00",5 out of 5 stars5 ratings
9,820.00,5 out of 5 stars58 ratings


In [15]:
#df["price"].astype(float)
#since non numeric, convert to pd.to_numeric and coerce errors to NaN, then fillna with 0
df["price"] = pd.to_numeric(df["price"], errors="coerce").fillna(0)

In [16]:
#create two columns: rating and rating count
#r"(\d+\.?\d*)\s*out of 5 stars\s*([\d,]+)"
#first regex extract group is the rating, second group is the rating count
#r"(\d+\.?\d*)
# \d+     one or more digits
#\.?     optional decimal point
#\d*     optional digits after decimal
#\s*out of 5 stars\s*([\d,]+)"
#\s*    optional spaces between the rating and the text "out of 5 stars" and between the text and the rating count
#[]     character set
#\d     digits
#,      comma allowed
#+      one or more
df[["rating","rating_count"]] = df["stars"].str.extract(r"(\d+\.?\d*)\s*out of 5 stars\s*([\d,]+)")
#since non numeric, convert to pd.to_numeric and coerce errors to NaN, then fillna with 0
df["rating_count"] = pd.to_numeric(df["rating_count"], errors="coerce").fillna(0)
#drop starts as its not needed
df = df.drop(columns=["stars"])

In [17]:
df[["rating","rating_count"]].head(105)

,rating,rating_count
0,5,34.0
1,4.5,41.0
2,4.5,38.0
3,4.5,12.0
4,4.5,181.0
...,...,...
100,NaN,0.0
101,NaN,0.0
102,4.5,21.0
103,5,6.0


In [18]:
df.iloc[100:105]

,name,author,narrator,releasedate,language,price,minutes,rating,rating_count
100,El monstruo marino secreto,Mattel,Vanessa Pérez Jurado,2022-01-04,spanish,76.0,30.0,NaN,0.0
101,Sabotage on the Solar Express,"M.G.Leonard,Sam Sedgman","Elisa Paganelli,Jot Davies",2022-02-17,english,323.0,335.0,NaN,0.0
102,A Boy Called Christmas,Matt Haig,Stephen Fry,2016-03-11,english,569.0,272.0,4.5,21.0
103,"The 39 Clues, Book 4",Jude Watson,David Pittu,2009-06-23,english,468.0,277.0,5,6.0
104,"The 39 Clues, Book 3",Peter Lerangis,David Pittu,2009-03-03,english,468.0,231.0,5,9.0


In [19]:
#Profile
Audible_Profile = ProfileReport(df, title="Audible Dataset EDA Profile")
Audible_Profile.to_file("Audible Profile.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 50.59it/s]


In [22]:
df.dtypes


name                    object
author                  object
narrator                object
releasedate     datetime64[ns]
language                object
price                  float64
minutes                float64
rating                  object
rating_count           float64
dtype: object

In [21]:
df.columns.tolist()


['name',
 'author',
 'narrator',
 'releasedate',
 'language',
 'price',
 'minutes',
 'rating',
 'rating_count']

In [ ]:
#visual graph of release date and genre


